# 16x_SHAP_candidate_interpretation_260516

12x model family comparison and 14x lightweight tuning outputs are used only to choose candidate models for model explanation. This notebook does not perform Optuna tuning, segmentation, feature removal, threshold selection, or causal analysis.

In [1]:
from pathlib import Path
import os
os.environ['LOKY_MAX_CPU_COUNT'] = os.environ.get('LOKY_MAX_CPU_COUNT', '8')
import json
import math
import shutil
import zipfile
import hashlib
import warnings
import importlib.util
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings('ignore', category=UserWarning)

STEP = '16x_SHAP_candidate_interpretation_260516'
RANDOM_STATE = 42
MAX_SHAP_SAMPLE = 5000

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / '.git').exists() and (candidate / 'park.ingyeom').exists():
            return candidate
    raise RuntimeError('repo root not found from notebook cwd')

REPO_ROOT = find_repo_root()
PARK = (REPO_ROOT / 'park.ingyeom').resolve()
NOTEBOOK_PATH = PARK / 'notebook' / STEP / f'{STEP}.ipynb'
OUT_DIR = PARK / 'reports' / 'interpretation' / STEP
FIG_DIR = PARK / 'reports' / 'figures' / STEP
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP}_review_package.zip'
for d in [OUT_DIR, FIG_DIR, ZIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def inside_park(path):
    path = Path(path).resolve()
    return path == PARK or PARK in path.parents

for path in [NOTEBOOK_PATH, OUT_DIR, FIG_DIR, ZIP_PATH]:
    if not inside_park(path):
        raise RuntimeError(f'output path outside park.ingyeom: {path}')

INPUTS = {
    '06x': PARK / 'reports' / 'audits' / '06x_dataset_generation_260515',
    '07x': PARK / 'reports' / 'audits' / '07x_feature_mapping_AARRR_260515',
    '10x': PARK / 'reports' / 'audits' / '10x_feature_distribution_redundancy_pre_audit_260516',
    '12x': PARK / 'reports' / 'models' / '12x_model_family_comparison_260516',
    '14x': PARK / 'reports' / 'models' / '14x_lightweight_candidate_tuning_260516',
}
REQUIRED = {
    '06x': ['06x_conservative_dataset.csv','06x_expanded_dataset.csv','06x_model_feature_lists.csv','06x_scope_feature_policy.csv','06x_caveat_register.csv','06x_final_checks.csv'],
    '07x': ['07x_feature_mapping_master.csv','07x_caveat_handoff.csv','07x_downstream_EDA_handoff.csv','07x_final_checks.csv'],
    '10x': ['10x_vif_pre_audit.csv','10x_pairwise_correlation_audit.csv','10x_redundancy_cluster_pre_audit.csv','10x_feature_refinement_candidate_policy.csv','10x_modeling_preflight_risk_register.csv','10x_downstream_handoff.csv','10x_final_checks.csv'],
    '12x': ['12x_model_summary_by_scope.csv','12x_candidate_selection_by_scope.csv','12x_oof_predictions.csv','12x_operating_metrics_at_k.csv','12x_calibration_decile_summary.csv','12x_redundancy_caveat_handoff.csv','12x_source_fingerprint_before_after.csv','12x_final_checks.csv'],
    '14x': ['14x_model_summary_by_scope.csv','14x_candidate_recommendation_summary.csv','14x_best_params_by_scope.csv','14x_oof_predictions.csv','14x_vs_12x_comparison.csv','14x_source_fingerprint_before_after.csv','14x_final_checks.csv'],
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def file_state(path):
    path = Path(path)
    if not path.exists():
        return {'sha256': '', 'size': np.nan, 'mtime': '', 'exists': False}
    st = path.stat()
    return {'sha256': sha256_file(path), 'size': st.st_size, 'mtime': datetime.fromtimestamp(st.st_mtime).isoformat(timespec='seconds'), 'exists': True}

source_files = [
    (INPUTS['06x'] / '06x_conservative_dataset.csv', '06x conservative dataset'),
    (INPUTS['06x'] / '06x_expanded_dataset.csv', '06x expanded dataset'),
    (INPUTS['06x'] / '06x_model_feature_lists.csv', '06x model feature list'),
    (INPUTS['07x'] / '07x_feature_mapping_master.csv', '07x feature mapping master'),
    (INPUTS['10x'] / '10x_vif_pre_audit.csv', '10x VIF pre-audit'),
    (INPUTS['12x'] / '12x_candidate_selection_by_scope.csv', '12x candidate selection'),
    (INPUTS['14x'] / '14x_candidate_recommendation_summary.csv', '14x candidate recommendation'),
    (INPUTS['14x'] / '14x_best_params_by_scope.csv', '14x best params'),
    (PARK / 'data' / '(광일)Membership_v2_with_derived_features.csv', 'raw source master if accessible'),
]
before_states = {str(p): file_state(p) for p, _ in source_files}

preflight_rows = []
for step, folder in INPUTS.items():
    preflight_rows.append({'item': f'{step}_folder_exists', 'status': 'PASS' if folder.exists() else 'FAIL', 'detail': str(folder), 'stop_reason': '' if folder.exists() else 'missing folder'})
    for fname in REQUIRED[step]:
        fpath = folder / fname
        preflight_rows.append({'item': f'{step}_{fname}_exists', 'status': 'PASS' if fpath.exists() else 'FAIL', 'detail': str(fpath), 'stop_reason': '' if fpath.exists() else 'missing required file'})
    hotfix = folder / '10x_hotfix_final_checks.csv'
    if step == '10x' and hotfix.exists():
        preflight_rows.append({'item': '10x_hotfix_final_checks_exists', 'status': 'PASS', 'detail': str(hotfix), 'stop_reason': ''})

def final_checks_pass(path):
    df = pd.read_csv(path)
    if 'status' not in df.columns:
        return False
    return (df['status'].astype(str).str.upper() == 'PASS').all()

for step in ['06x','07x','10x','12x','14x']:
    fpath = INPUTS[step] / f'{step}_final_checks.csv'
    ok = fpath.exists() and final_checks_pass(fpath)
    preflight_rows.append({'item': f'{step}_final_checks_pass', 'status': 'PASS' if ok else 'FAIL', 'detail': str(fpath), 'stop_reason': '' if ok else 'final checks not all PASS'})
hotfix_path = INPUTS['10x'] / '10x_hotfix_final_checks.csv'
if hotfix_path.exists():
    ok = final_checks_pass(hotfix_path)
    preflight_rows.append({'item': '10x_hotfix_final_checks_pass', 'status': 'PASS' if ok else 'FAIL', 'detail': str(hotfix_path), 'stop_reason': '' if ok else 'hotfix checks not all PASS'})

shap_spec = importlib.util.find_spec('shap')
preflight_rows.append({'item': 'shap_import_available', 'status': 'PASS' if shap_spec else 'FAIL', 'detail': str(shap_spec.origin) if shap_spec else '', 'stop_reason': '' if shap_spec else 'shap package unavailable'})
if not shap_spec:
    pd.DataFrame(preflight_rows).to_csv(OUT_DIR / '16x_preflight_input_validation.csv', index=False, encoding='utf-8-sig')
    raise RuntimeError('SHAP unavailable. Stopping per 16x policy.')
import shap

preflight_rows.append({'item': 'output_folder_inside_park_ingyeom', 'status': 'PASS' if inside_park(OUT_DIR) and inside_park(FIG_DIR) else 'FAIL', 'detail': f'{OUT_DIR} | {FIG_DIR}', 'stop_reason': ''})
pd.DataFrame(preflight_rows).to_csv(OUT_DIR / '16x_preflight_input_validation.csv', index=False, encoding='utf-8-sig')
if any(r['status'] == 'FAIL' for r in preflight_rows):
    raise RuntimeError('preflight failed; see 16x_preflight_input_validation.csv')

def truthy(v):
    return str(v).strip().lower() in ['true','1','yes','y','pass']

conservative_df = pd.read_csv(INPUTS['06x'] / '06x_conservative_dataset.csv')
expanded_df = pd.read_csv(INPUTS['06x'] / '06x_expanded_dataset.csv')
feature_audit = pd.read_csv(INPUTS['12x'] / '12x_feature_set_input_audit.csv')
mapping = pd.read_csv(INPUTS['07x'] / '07x_feature_mapping_master.csv')
vif = pd.read_csv(INPUTS['10x'] / '10x_vif_pre_audit.csv')
clusters = pd.read_csv(INPUTS['10x'] / '10x_redundancy_cluster_pre_audit.csv')
corr = pd.read_csv(INPUTS['10x'] / '10x_pairwise_correlation_audit.csv')
rec14 = pd.read_csv(INPUTS['14x'] / '14x_candidate_recommendation_summary.csv')
params14 = pd.read_csv(INPUTS['14x'] / '14x_best_params_by_scope.csv')
cand12 = pd.read_csv(INPUTS['12x'] / '12x_candidate_selection_by_scope.csv')
model_summary12 = pd.read_csv(INPUTS['12x'] / '12x_model_summary_by_scope.csv')
model_summary14 = pd.read_csv(INPUTS['14x'] / '14x_model_summary_by_scope.csv')

font_rows = []
font_candidates = ['Malgun Gothic', 'Noto Sans CJK KR', 'Noto Sans KR', 'AppleGothic']
available_fonts = {f.name: f.fname for f in font_manager.fontManager.ttflist}
selected_font = None
for name in font_candidates:
    found = name in available_fonts
    if found and selected_font is None:
        selected_font = name
    font_rows.append({'candidate_font': name, 'found': 'yes' if found else 'no', 'selected': 'yes' if selected_font == name else 'no', 'matplotlib_font_family': selected_font or '', 'unicode_minus_setting': False, 'status': 'PASS' if found else 'WARN', 'note': available_fonts.get(name, 'not found')})
if selected_font:
    plt.rcParams['font.family'] = selected_font
else:
    font_rows.append({'candidate_font': 'fallback_default', 'found': 'yes', 'selected': 'yes', 'matplotlib_font_family': 'default', 'unicode_minus_setting': False, 'status': 'WARN', 'note': 'Korean font not found; English label fallback used'})
plt.rcParams['axes.unicode_minus'] = False
pd.DataFrame(font_rows).to_csv(OUT_DIR / '16x_font_preflight.csv', index=False, encoding='utf-8-sig')

visual_inventory = []
visual_warnings = []
def save_fig(path, fig_type, feature_set='', scope='', model='', note=''):
    path = Path(path)
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.close()
    visual_inventory.append({'figure_path': str(path), 'figure_type': fig_type, 'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model, 'created': 'yes' if path.exists() else 'no', 'file_size': path.stat().st_size if path.exists() else 0, 'status': 'PASS' if path.exists() and path.stat().st_size > 0 else 'FAIL', 'note': note})

plt.figure(figsize=(8, 3))
label = '한글 폰트 테스트: 재구매 / 이탈위험 / 중요도' if selected_font else 'Korean font test: repurchase / churn risk / importance'
plt.title(label)
plt.bar(['재구매', '이탈위험', '중요도'] if selected_font else ['repurchase','churn risk','importance'], [1, 2, 3], color=['#1D9E75','#D4537E','#378ADD'])
plt.ylabel('값' if selected_font else 'value')
save_fig(FIG_DIR / '16x_fig_00_korean_font_test.png', 'font_test', note='selected_font=' + str(selected_font))

def dataset_for_feature_set(feature_set):
    return expanded_df if feature_set == 'expanded_feature_set' else conservative_df

def scope_filter(df, scope):
    if scope == 'promotion_only':
        return df[df['is_promotion'] == 1].copy()
    if scope == 'nonpromotion_only':
        return df[df['is_promotion'] == 0].copy()
    return df.copy()

def features_for(feature_set, scope):
    sub = feature_audit[(feature_audit['feature_set_name'] == feature_set) & (feature_audit['dataset_scope'] == scope)].copy()
    sub = sub[sub['included_as_feature'].astype(str).str.lower() == 'true']
    return sub['feature_name'].tolist()

preferred = [
    ('expanded_feature_set','overall_with_promotion'),
    ('expanded_feature_set','overall_without_promotion'),
    ('expanded_feature_set','promotion_only'),
    ('expanded_feature_set','nonpromotion_only'),
]
plan_rows = []
for feature_set, scope in preferred:
    sub14 = rec14[(rec14['feature_set_name'] == feature_set) & (rec14['dataset_scope'] == scope)].copy()
    sub14['candidate_rank'] = sub14['tuned_model'].map({'LightGBM': 1, 'CatBoost': 2, 'HistGradientBoosting': 3, 'XGBoost': 4}).fillna(9)
    ok14 = sub14[sub14['use_for_16x_SHAP_candidate'].astype(str).str.contains('yes|review_candidate', case=False, na=False)].sort_values('candidate_rank')
    if len(ok14):
        row = ok14.iloc[0]
        source_step = '14x_tuned'
        model_name = row['tuned_model']
        basis = f"14x use_for_16x_SHAP_candidate={row['use_for_16x_SHAP_candidate']}; {row['recommendation']}"
    else:
        row = cand12[(cand12['feature_set_name'] == feature_set) & (cand12['dataset_scope'] == scope)].iloc[0]
        source_step = '12x_fixed'
        model_name = row['recommended_for_16x_SHAP']
        basis = '14x candidate unavailable or not suitable; fallback to 12x recommended_for_16x_SHAP'
    scoped = scope_filter(dataset_for_feature_set(feature_set), scope)
    feats = features_for(feature_set, scope)
    plan_rows.append({'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'source_step': source_step, 'selection_basis': basis, 'row_count': len(scoped), 'feature_count': len(feats), 'will_run_SHAP': 'yes', 'reason': 'priority expanded scope candidate for 16x SHAP'})

feature_set, scope = 'conservative_safe_22', 'overall_with_promotion'
row = cand12[(cand12['feature_set_name'] == feature_set) & (cand12['dataset_scope'] == scope)].iloc[0]
scoped = scope_filter(dataset_for_feature_set(feature_set), scope)
feats = features_for(feature_set, scope)
plan_rows.append({'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': row['recommended_for_16x_SHAP'], 'source_step': '12x_fixed', 'selection_basis': 'conservative_safe_22 reference: minimum one overall scope; 12x recommended_for_16x_SHAP used', 'row_count': len(scoped), 'feature_count': len(feats), 'will_run_SHAP': 'yes', 'reason': 'comparison reference only, not final model'})
candidate_plan = pd.DataFrame(plan_rows)
candidate_plan.to_csv(OUT_DIR / '16x_SHAP_candidate_plan.csv', index=False, encoding='utf-8-sig')

def parse_json_params(s):
    if pd.isna(s) or str(s).strip() == '':
        return {}
    return json.loads(str(s))

def fixed_params(model_name):
    if model_name == 'LightGBM':
        return {'n_estimators': 120, 'learning_rate': 0.05, 'num_leaves': 31, 'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_lambda': 1.0}
    if model_name == 'CatBoost':
        return {'iterations': 120, 'depth': 4, 'learning_rate': 0.05}
    if model_name == 'HistGradientBoosting':
        return {'max_iter': 100, 'learning_rate': 0.06, 'max_leaf_nodes': 31, 'l2_regularization': 0.0}
    if model_name == 'XGBoost':
        return {'n_estimators': 120, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 0.9, 'colsample_bytree': 0.9, 'eval_metric': 'logloss'}
    return {}

def build_model(model_name, params):
    if model_name == 'LightGBM':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1, **params)
    if model_name == 'CatBoost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(loss_function='Logloss', eval_metric='AUC', random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False, **params)
    if model_name == 'HistGradientBoosting':
        return HistGradientBoostingClassifier(random_state=RANDOM_STATE, **params)
    if model_name == 'XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, **params)
    raise ValueError(f'unsupported model for 16x SHAP: {model_name}')

def prepare_X(df, features):
    X = df[features].copy()
    for col in X.columns:
        if X[col].dtype == 'bool':
            X[col] = X[col].astype(int)
        elif not pd.api.types.is_numeric_dtype(X[col]):
            X[col] = pd.to_numeric(X[col], errors='coerce')
    return X.fillna(0)

def select_sample(df):
    n = len(df)
    if n <= MAX_SHAP_SAMPLE:
        return df.index.to_numpy(), 'full_data_used_because_row_count_le_5000'
    key = df['is_repurchase'].astype(str)
    if 'is_promotion' in df.columns:
        key = key + '_' + df['is_promotion'].astype(str)
    counts = key.value_counts()
    stratify = key if counts.min() >= 2 else None
    sample_idx, _ = train_test_split(df.index.to_numpy(), train_size=MAX_SHAP_SAMPLE, random_state=RANDOM_STATE, stratify=stratify)
    return np.array(sample_idx), 'deterministic_stratified_sample_max_5000' if stratify is not None else 'deterministic_random_sample_max_5000'

map_cols = ['feature_set_name','safe_model_feature_name','feature_family','AARRR_stage','caveat_flag','caveat_reason']
mapping_small = mapping[map_cols].drop_duplicates()
vif_small = vif[['feature_set_name','safe_model_feature_name','vif_risk_bucket','redundancy_review_needed']].drop_duplicates()
feature_meta = feature_audit[['feature_set_name','dataset_scope','feature_name','redundancy_family_from_10x','caveat_flag','caveat_reason']].drop_duplicates()

sample_rows = []
sample_ids = []
fit_rows = []
global_rows = []
family_rows = []
direction_rows = []

def shap_array_for(model, X_sample):
    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(X_sample)
    if isinstance(values, list):
        values = values[1] if len(values) > 1 else values[0]
    values = np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, 1] if values.shape[2] > 1 else values[:, :, 0]
    return values

def slug_model(name):
    return str(name).replace(' ', '')

for _, cand in candidate_plan.iterrows():
    feature_set = cand['feature_set_name']
    scope = cand['dataset_scope']
    model_name = cand['model_name']
    source_step = cand['source_step']
    df_full = scope_filter(dataset_for_feature_set(feature_set), scope)
    features = features_for(feature_set, scope)
    y = df_full['is_repurchase'].astype(int)
    X = prepare_X(df_full, features)
    if source_step == '14x_tuned':
        psub = params14[(params14['feature_set_name'] == feature_set) & (params14['dataset_scope'] == scope) & (params14['model_name'] == model_name)]
        params = parse_json_params(psub.iloc[0]['best_params_json']) if len(psub) else fixed_params(model_name)
        params_source = '14x_best_params_by_scope' if len(psub) else '12x_fixed_fallback_due_missing_14x_params'
    else:
        params = fixed_params(model_name)
        params_source = '12x_fixed_parameter_spec_from_12x_notebook'
    model = build_model(model_name, params)
    try:
        model.fit(X, y)
        fit_status = 'PASS'
        note = 'fitted full scope data for candidate model explanation'
    except Exception as e:
        fit_status = 'FAIL'
        note = repr(e)
        fit_rows.append({'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'params_source': params_source, 'params_json': json.dumps(params, ensure_ascii=False, sort_keys=True), 'fit_row_count': len(X), 'feature_count': len(features), 'fit_status': fit_status, 'note': note})
        raise
    fit_rows.append({'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'params_source': params_source, 'params_json': json.dumps(params, ensure_ascii=False, sort_keys=True), 'fit_row_count': len(X), 'feature_count': len(features), 'fit_status': fit_status, 'note': note})
    sample_idx, sample_method = select_sample(df_full)
    sample_df = df_full.loc[sample_idx].copy()
    X_sample = X.loc[sample_idx].copy()
    shap_values = shap_array_for(model, X_sample)
    sample_rows.append({'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'full_row_count': len(df_full), 'sample_row_count': len(sample_df), 'target_rate_full': y.mean(), 'target_rate_sample': sample_df['is_repurchase'].mean(), 'promotion_rate_full_if_applicable': df_full['is_promotion'].mean() if 'is_promotion' in df_full else np.nan, 'promotion_rate_sample_if_applicable': sample_df['is_promotion'].mean() if 'is_promotion' in sample_df else np.nan, 'sample_method': sample_method, 'random_state': RANDOM_STATE})
    for ridx in sample_idx:
        sample_ids.append({'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'source_row_index': int(ridx), 'USER_KEY': sample_df.loc[ridx, 'USER_KEY'] if 'USER_KEY' in sample_df.columns else ''})
    mean_abs = np.abs(shap_values).mean(axis=0)
    mean_signed = shap_values.mean(axis=0)
    imp = pd.DataFrame({'feature_name': features, 'mean_abs_shap': mean_abs, 'mean_shap': mean_signed}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    imp['rank'] = np.arange(1, len(imp) + 1)
    imp['feature_set_name'] = feature_set
    imp['dataset_scope'] = scope
    imp['model_name'] = model_name
    imp = imp.merge(mapping_small, left_on=['feature_set_name','feature_name'], right_on=['feature_set_name','safe_model_feature_name'], how='left')
    imp = imp.merge(feature_meta, on=['feature_set_name','dataset_scope','feature_name'], how='left', suffixes=('_07x','_12x'))
    imp['feature_family'] = imp['feature_family'].fillna('unmapped')
    imp['AARRR_stage'] = imp['AARRR_stage'].fillna('unmapped')
    imp['redundancy_family'] = imp['redundancy_family_from_10x'].fillna('not_clustered')
    imp['caveat_flag'] = imp['caveat_flag_12x'].fillna(imp['caveat_flag_07x']).fillna(False)
    imp['caveat_reason'] = imp['caveat_reason_12x'].fillna(imp['caveat_reason_07x']).fillna('')
    keep_cols = ['feature_set_name','dataset_scope','model_name','feature_name','mean_abs_shap','mean_shap','rank','feature_family','AARRR_stage','redundancy_family','caveat_flag','caveat_reason']
    global_rows.append(imp[keep_cols])
    fam = imp.groupby(['feature_set_name','dataset_scope','model_name','feature_family','AARRR_stage','redundancy_family'], dropna=False).agg(mean_abs_shap_sum=('mean_abs_shap','sum'), mean_abs_shap_mean=('mean_abs_shap','mean'), feature_count=('feature_name','count'), top_features=('feature_name', lambda s: '; '.join(list(s.head(5))))).reset_index()
    fam['interpretation_caution'] = 'Prefer family-level reading because correlated features can split SHAP importance; no feature removal was performed.'
    family_rows.append(fam)
    top_for_direction = imp.head(20)['feature_name'].tolist()
    for feat in top_for_direction:
        vals = X_sample[feat]
        shap_col = shap_values[:, features.index(feat)]
        q25, q75 = vals.quantile(0.25), vals.quantile(0.75)
        low_mask = vals <= q25
        high_mask = vals >= q75
        high_mean = float(np.mean(shap_col[high_mask])) if high_mask.any() else np.nan
        low_mean = float(np.mean(shap_col[low_mask])) if low_mask.any() else np.nan
        assoc = 'higher_feature_values_associated_with_higher_repurchase_score' if high_mean > low_mean else 'higher_feature_values_associated_with_lower_repurchase_score'
        direction_rows.append({'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'feature_name': feat, 'high_feature_value_association': assoc, 'low_feature_value_association': 'opposite_direction_in_this_fitted_model', 'mean_shap_high_value': high_mean, 'mean_shap_low_value': low_mean, 'direction_interpretation': assoc.replace('_', ' ') + '; model explanation for repurchase_score only', 'caution': 'Do not read this as causality or as a churn intervention effect.'})
    stem = f'{feature_set}__{scope}__{slug_model(model_name)}'
    try:
        shap.summary_plot(shap_values, X_sample, show=False, max_display=20)
        save_fig(FIG_DIR / f'16x_fig_SHAP_beeswarm__{stem}.png', 'SHAP_beeswarm', feature_set, scope, model_name)
    except Exception as e:
        visual_warnings.append({'figure_type': 'SHAP_beeswarm', 'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'warning': repr(e), 'action': 'required figure failed'})
    try:
        shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False, max_display=20)
        save_fig(FIG_DIR / f'16x_fig_SHAP_bar__{stem}.png', 'SHAP_bar', feature_set, scope, model_name)
    except Exception as e:
        visual_warnings.append({'figure_type': 'SHAP_bar', 'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'warning': repr(e), 'action': 'required figure failed'})
    try:
        fam_plot = fam.sort_values('mean_abs_shap_sum', ascending=False).head(15).sort_values('mean_abs_shap_sum')
        plt.figure(figsize=(9, max(4, 0.38 * len(fam_plot))))
        plt.barh(fam_plot['feature_family'].astype(str), fam_plot['mean_abs_shap_sum'], color='#378ADD')
        plt.title(f'Feature family SHAP importance: {feature_set} / {scope} / {model_name}')
        plt.xlabel('sum mean(|SHAP|), repurchase_score direction')
        save_fig(FIG_DIR / f'16x_fig_SHAP_family_bar__{stem}.png', 'SHAP_family_bar', feature_set, scope, model_name)
    except Exception as e:
        visual_warnings.append({'figure_type': 'SHAP_family_bar', 'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'warning': repr(e), 'action': 'required figure failed'})
    for rank_i, feat in enumerate(imp.head(5)['feature_name'].tolist(), start=1):
        try:
            shap.dependence_plot(feat, shap_values, X_sample, show=False)
            save_fig(FIG_DIR / f'16x_fig_SHAP_dependence_top{rank_i:02d}__{stem}.png', 'SHAP_dependence', feature_set, scope, model_name)
        except Exception as e:
            visual_warnings.append({'figure_type': f'SHAP_dependence_top{rank_i:02d}', 'feature_set_name': feature_set, 'dataset_scope': scope, 'model_name': model_name, 'warning': repr(e), 'action': 'optional dependence warning recorded'})

sample_audit = pd.DataFrame(sample_rows)
sample_audit.to_csv(OUT_DIR / '16x_SHAP_sample_audit.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(sample_ids).to_csv(OUT_DIR / '16x_SHAP_sample_row_ids.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(fit_rows).to_csv(OUT_DIR / '16x_model_refit_summary.csv', index=False, encoding='utf-8-sig')
global_importance = pd.concat(global_rows, ignore_index=True)
global_importance.to_csv(OUT_DIR / '16x_SHAP_global_importance.csv', index=False, encoding='utf-8-sig')
family_importance = pd.concat(family_rows, ignore_index=True)
family_importance.to_csv(OUT_DIR / '16x_SHAP_family_importance.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(direction_rows).to_csv(OUT_DIR / '16x_SHAP_direction_summary.csv', index=False, encoding='utf-8-sig')

scope_comp_rows = []
exp_global = global_importance[global_importance['feature_set_name'] == 'expanded_feature_set'].copy()
for _, r in exp_global[exp_global['rank'] <= 10].iterrows():
    scopes_with_feature = exp_global[(exp_global['feature_name'] == r['feature_name']) & (exp_global['rank'] <= 10)]['dataset_scope'].nunique()
    scope_comp_rows.append({'feature_set_name': r['feature_set_name'], 'dataset_scope': r['dataset_scope'], 'model_name': r['model_name'], 'rank': r['rank'], 'feature_name': r['feature_name'], 'mean_abs_shap': r['mean_abs_shap'], 'shared_across_scopes': 'yes' if scopes_with_feature > 1 else 'no', 'notes': 'top10 SHAP feature within expanded feature-set candidate scopes'})
scope_comp = pd.DataFrame(scope_comp_rows)
scope_comp.to_csv(OUT_DIR / '16x_scope_comparison_top_features.csv', index=False, encoding='utf-8-sig')

try:
    scopes = [s for s in ['overall_with_promotion','overall_without_promotion','promotion_only','nonpromotion_only'] if s in exp_global['dataset_scope'].unique()]
    fig, axes = plt.subplots(len(scopes), 1, figsize=(10, max(7, 2.5 * len(scopes))))
    if len(scopes) == 1:
        axes = [axes]
    for ax, scope in zip(axes, scopes):
        sub = exp_global[(exp_global['dataset_scope'] == scope) & (exp_global['rank'] <= 10)].sort_values('mean_abs_shap')
        ax.barh(sub['feature_name'], sub['mean_abs_shap'], color='#378ADD')
        model_label = sub['model_name'].iloc[0] if len(sub) else ''
        ax.set_title(f'{scope} / {model_label}')
        ax.set_xlabel('mean(|SHAP|)')
    fig.suptitle('Expanded feature-set top 10 SHAP comparison by scope')
    save_fig(FIG_DIR / '16x_fig_scope_top10_SHAP_comparison.png', 'scope_top10_SHAP_comparison', 'expanded_feature_set', 'all_priority_scopes', 'mixed')
except Exception as e:
    visual_warnings.append({'figure_type': 'scope_top10_SHAP_comparison', 'feature_set_name': 'expanded_feature_set', 'dataset_scope': 'all_priority_scopes', 'model_name': 'mixed', 'warning': repr(e), 'action': 'required figure failed'})

cluster_rows = []
for _, cl in clusters.iterrows():
    features_in_family = [x.strip() for x in str(cl['cluster_feature_list']).split(';') if x.strip()]
    sub = global_importance[(global_importance['feature_set_name'] == cl['feature_set_name']) & (global_importance['feature_name'].isin(features_in_family))].copy()
    top_feats = '; '.join(sub.sort_values('mean_abs_shap', ascending=False).head(5)['feature_name'].drop_duplicates().tolist())
    cluster_rows.append({'redundancy_family': cl['cluster_id'], 'features_in_family': cl['cluster_feature_list'], 'high_vif_or_corr_flag': 'yes', 'top_SHAP_features_in_family': top_feats, 'interpretation_rule': 'Interpret as a related feature family or redundancy cluster; do not over-read one variable.', 'user_approval_required_for_removal': 'yes', 'removal_performed': 'no'})
redundancy_audit = pd.DataFrame(cluster_rows)
redundancy_audit.to_csv(OUT_DIR / '16x_redundancy_SHAP_caveat_audit.csv', index=False, encoding='utf-8-sig')
try:
    red_plot = global_importance.groupby('redundancy_family', dropna=False)['mean_abs_shap'].sum().sort_values(ascending=False).head(20).sort_values()
    plt.figure(figsize=(10, max(5, 0.35 * len(red_plot))))
    plt.barh(red_plot.index.astype(str), red_plot.values, color='#D4537E')
    plt.title('Redundancy family SHAP importance, all 16x candidates')
    plt.xlabel('sum mean(|SHAP|)')
    save_fig(FIG_DIR / '16x_fig_redundancy_family_SHAP_importance.png', 'redundancy_family_SHAP_importance', 'all', 'all', 'mixed')
except Exception as e:
    visual_warnings.append({'figure_type': 'redundancy_family_SHAP_importance', 'feature_set_name': 'all', 'dataset_scope': 'all', 'model_name': 'mixed', 'warning': repr(e), 'action': 'required figure failed'})

top_family = family_importance.sort_values('mean_abs_shap_sum', ascending=False).head(8)
business_rows = []
for i, r in enumerate(top_family.itertuples(index=False), start=1):
    business_rows.append({'insight_id': f'SHAP_FAMILY_{i:02d}', 'evidence_type': 'SHAP', 'feature_or_family': r.feature_family, 'interpretation': 'This fitted candidate model assigns relatively large mean absolute SHAP contribution to this feature family.', 'business_implication': 'Use as interpretation candidate only, with EDA and redundancy caveats reviewed together.', 'safe_wording': '이 모델에서 해당 feature family가 repurchase_score 설명에 크게 기여했다.', 'unsafe_wording': '이 feature family가 이탈의 원인이다.', 'caveat': 'Model explanation, not causality; correlated features may split importance.', 'use_in_presentation_candidate': 'yes'})
business_rows += [
    {'insight_id': 'EDA_LINK_01', 'evidence_type': 'EDA', 'feature_or_family': 'week3 usage and retention behavior', 'interpretation': 'EDA and SHAP can be read together when week-3 usage signals appear in both.', 'business_implication': 'Presentation wording should say common signal, not intervention effect.', 'safe_wording': 'EDA와 SHAP에서 공통적으로 3주차 사용량 관련 신호가 강하게 나타났다.', 'unsafe_wording': '3주차 사용량을 늘리면 재구매가 오른다.', 'caveat': 'Descriptive and model-explanation evidence only.', 'use_in_presentation_candidate': 'yes'},
    {'insight_id': 'REDUNDANCY_10X_01', 'evidence_type': '10x', 'feature_or_family': 'high VIF / correlated clusters', 'interpretation': 'Redundant features should be interpreted as groups.', 'business_implication': 'Avoid single-feature overclaim.', 'safe_wording': 'VIF/redundancy 때문에 family 단위 해석을 우선한다.', 'unsafe_wording': '상관 높은 변수 하나만 제거하면 된다.', 'caveat': 'No feature removal was performed.', 'use_in_presentation_candidate': 'yes'},
    {'insight_id': 'MODEL_12X_01', 'evidence_type': '12x', 'feature_or_family': 'candidate comparison', 'interpretation': '12x provides candidate context, not a final model.', 'business_implication': 'AUC must be read with operating and calibration diagnostics.', 'safe_wording': '12x는 후보 비교 근거이며 최종 모델 확정은 아니다.', 'unsafe_wording': '12x 최고 AUC 모델이 최종 모델이다.', 'caveat': 'No threshold selected.', 'use_in_presentation_candidate': 'yes'},
    {'insight_id': 'TUNING_14X_01', 'evidence_type': '14x', 'feature_or_family': 'lightweight tuning candidates', 'interpretation': '14x tuned parameters are used to refit candidate models for explanation.', 'business_implication': 'Tuning sensitivity informs SHAP candidate choice only.', 'safe_wording': '14x 후보 기반으로 SHAP 해석 대상을 정했다.', 'unsafe_wording': '14x 튜닝 결과가 최종 모델이다.', 'caveat': 'No Optuna was run in 16x.', 'use_in_presentation_candidate': 'yes'},
]
pd.DataFrame(business_rows).to_csv(OUT_DIR / '16x_business_interpretation_candidates.csv', index=False, encoding='utf-8-sig')

safe_unsafe = [
    ('unsafe','SHAP이 원인을 밝혔다','SHAP은 fitted model의 repurchase_score 설명 근거다.'),
    ('unsafe','이 feature를 바꾸면 재구매가 오른다','이 모델에서 feature value와 SHAP direction이 repurchase_score 방향으로 연결되어 보인다.'),
    ('unsafe','100원딜 때문에 이탈했다','promotion 관련 feature는 모델 설명 신호일 뿐 인과효과가 아니다.'),
    ('unsafe','top10 churn_risk가 캠페인 대상이다','top-k와 churn_risk는 진단 지표이며 campaign threshold는 아직 아니다.'),
    ('unsafe','최종 모델 확정','16x는 후보 모델 해석 단계이며 최종 모델 확정이 아니다.'),
    ('unsafe','unique user 기준','분석 단위는 row-level / subscription-event-level이다.'),
    ('safe','이 모델에서 해당 feature/family가 repurchase_score 설명에 크게 기여했다',''),
    ('safe','EDA와 SHAP에서 공통적으로 3주차 사용량 관련 신호가 강하게 나타났다',''),
    ('safe','해석은 모델 설명이며 인과가 아니다',''),
]
pd.DataFrame(safe_unsafe, columns=['wording_type','wording','safe_alternative']).to_csv(OUT_DIR / '16x_safe_unsafe_wording.csv', index=False, encoding='utf-8-sig')

risks = [
    ('SHAP is model explanation not causality','open','Keep causal wording out of 16x and presentation.'),
    ('high VIF / redundancy','open','Prefer feature family and redundancy family interpretation.'),
    ('feature family interpretation preferred','open','Do not overinterpret one variable inside correlated clusters.'),
    ('is_churn_prevented interpretation caution','open','Treat as policy/caveat-sensitive feature, not causal proof.'),
    ('content/genre caveat','open','Genre/content ratios may be proxies and should not be read as direct preference causes.'),
    ('old_movie_ratio_5y 9-row caveat','open','Keep small-row caveat visible when used in interpretation.'),
    ('final segment threshold not selected','open','Leave segmentation design to 17x.'),
    ('campaign threshold not selected','open','Do not convert top-k diagnostics into campaign target threshold.'),
]
pd.DataFrame(risks, columns=['risk','status','next_step']).to_csv(OUT_DIR / '16x_open_risks_for_next_steps.csv', index=False, encoding='utf-8-sig')

pd.DataFrame(visual_inventory).to_csv(OUT_DIR / '16x_visualization_inventory.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(visual_warnings, columns=['figure_type','feature_set_name','dataset_scope','model_name','warning','action']).to_csv(OUT_DIR / '16x_visualization_warnings.csv', index=False, encoding='utf-8-sig')

after_rows = []
for p, role in source_files:
    b = before_states[str(p)]
    a = file_state(p)
    status = 'UNCHANGED' if b['sha256'] == a['sha256'] and b['size'] == a['size'] and b['exists'] == a['exists'] else 'CHANGED'
    if not b['exists'] and not a['exists']:
        status = 'MISSING_OPTIONAL'
    after_rows.append({'file_path': str(p), 'file_role': role, 'sha256_before': b['sha256'], 'sha256_after': a['sha256'], 'size_before': b['size'], 'size_after': a['size'], 'mtime_before': b['mtime'], 'mtime_after': a['mtime'], 'status': status})
fingerprint = pd.DataFrame(after_rows)
fingerprint.to_csv(OUT_DIR / '16x_source_fingerprint_before_after.csv', index=False, encoding='utf-8-sig')

top_features_summary = global_importance.sort_values('mean_abs_shap', ascending=False).head(12)
top_family_summary = family_importance.sort_values('mean_abs_shap_sum', ascending=False).head(8)
readme_lines = []
readme_lines.append(f'# {STEP}')
readme_lines.append('')
readme_lines.append('Purpose: SHAP-based model interpretation for candidate models selected from 12x model-family comparison and 14x lightweight tuning. This is not final model selection, Optuna, segmentation, feature removal, thresholding, or causal analysis.')
readme_lines.append('')
readme_lines.append('Interpretation unit: row-level / subscription-event-level. Do not describe this as unique-user analysis.')
readme_lines.append('')
readme_lines.append('Class direction: target=is_repurchase, positive class=is_repurchase=1, repurchase_score=P(is_repurchase=1), churn_risk=1-repurchase_score. SHAP signs are interpreted only as directions for repurchase_score.')
readme_lines.append('')
readme_lines.append('Important wording: model explanation on fitted candidate model, not causal and not OOF explanation.')
readme_lines.append('')
readme_lines.append('## Input paths')
for step, folder in INPUTS.items():
    readme_lines.append(f'- {step}: {folder}')
readme_lines.append('')
readme_lines.append('## Candidate models')
for r in candidate_plan.itertuples(index=False):
    readme_lines.append(f'- {r.feature_set_name} / {r.dataset_scope} / {r.model_name}: {r.source_step}; {r.selection_basis}')
readme_lines.append('')
readme_lines.append(f'SHAP sample policy: if scope row count <= {MAX_SHAP_SAMPLE}, use all rows; otherwise deterministic stratified sample up to {MAX_SHAP_SAMPLE} rows with random_state={RANDOM_STATE}. Sample row IDs are saved in 16x_SHAP_sample_row_ids.csv.')
readme_lines.append('')
readme_lines.append(f'Korean font setting: selected_font={selected_font or "none"}; axes.unicode_minus=False. Font test figure: 16x_fig_00_korean_font_test.png.')
readme_lines.append('')
readme_lines.append('## Main SHAP results')
for r in top_features_summary.itertuples(index=False):
    readme_lines.append(f'- {r.feature_set_name} / {r.dataset_scope} / {r.model_name}: rank {int(r.rank)} {r.feature_name}, family={r.feature_family}, mean_abs_shap={r.mean_abs_shap:.6f}')
readme_lines.append('')
readme_lines.append('## Feature family interpretation')
for r in top_family_summary.itertuples(index=False):
    readme_lines.append(f'- {r.feature_set_name} / {r.dataset_scope}: {r.feature_family}, AARRR={r.AARRR_stage}, redundancy_family={r.redundancy_family}, sum_mean_abs_shap={r.mean_abs_shap_sum:.6f}, top_features={r.top_features}')
readme_lines.append('')
readme_lines.append('## 10x redundancy caveat')
readme_lines.append('High VIF and correlation clusters mean individual feature importance can be split across related variables. Use feature family or redundancy cluster wording; no feature removal was performed and user approval remains required for removal decisions.')
readme_lines.append('')
readme_lines.append('## EDA/modeling alignment')
readme_lines.append('Use SHAP together with 10x redundancy diagnostics, 12x candidate comparison, 14x tuning sensitivity, and prior EDA. A repeated week-3 usage or retention signal can be described as a common descriptive/model-explanation signal, not as an intervention effect.')
readme_lines.append('')
readme_lines.append('## Forbidden interpretations')
readme_lines.append('- Do not say SHAP identified causes.')
readme_lines.append('- Do not say changing one feature will raise repurchase.')
readme_lines.append('- Do not say promotion caused churn or retention.')
readme_lines.append('- Do not use top-k churn_risk as a campaign threshold.')
readme_lines.append('- Do not call this final model selection or unique-user analysis.')
readme_lines.append('')
readme_lines.append('Next step: 17x segmentation design.')
(OUT_DIR / 'README.md').write_text('\n'.join(readme_lines) + '\n', encoding='utf-8')

note_path = PARK / 'note.md'
note_text = note_path.read_text(encoding='utf-8') if note_path.exists() else ''
note_add = []
note_add.append('')
note_add.append(f'## {STEP}')
note_add.append(f'- 수행: 12x/14x 후보 기반 SHAP 해석을 완료했다. SHAP은 인과가 아니라 fitted candidate model의 repurchase_score model explanation이다.')
note_add.append('- 사용 후보: ' + '; '.join([f"{r.feature_set_name}/{r.dataset_scope}/{r.model_name}" for r in candidate_plan.itertuples(index=False)]))
note_add.append(f'- 한글 폰트 설정: selected_font={selected_font or "none"}, axes.unicode_minus=False, font test figure 생성 완료.')
note_add.append('- 주요 top feature/family는 16x_SHAP_global_importance.csv와 16x_SHAP_family_importance.csv에 기록했다.')
note_add.append('- VIF/redundancy 때문에 개별 변수보다 feature family/redundancy family 단위 해석을 권장한다. feature removal은 수행하지 않았다.')
note_add.append('- 최종 segmentation/threshold/campaign threshold는 아직 아니다. 다음 단계는 17x segmentation design이다.')
if f'## {STEP}' not in note_text:
    note_path.write_text(note_text.rstrip() + '\n' + '\n'.join(note_add) + '\n', encoding='utf-8')
tail = '\n'.join(note_path.read_text(encoding='utf-8').splitlines()[-160:]) + '\n'
(OUT_DIR / 'note_tail_copy.md').write_text(tail, encoding='utf-8')

def exists_nonempty(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0

fig_inv = pd.read_csv(OUT_DIR / '16x_visualization_inventory.csv')
required_figs = ['font_test','SHAP_beeswarm','SHAP_bar','SHAP_family_bar','scope_top10_SHAP_comparison','redundancy_family_SHAP_importance']
required_figs_ok = all((fig_inv[fig_inv['figure_type'] == t]['status'].astype(str).str.upper() == 'PASS').any() for t in required_figs)
source_unchanged = fingerprint['status'].isin(['UNCHANGED','MISSING_OPTIONAL']).all()
checks = []
def check(name, ok, detail=''):
    checks.append({'check': name, 'status': 'PASS' if ok else 'FAIL', 'detail': detail})
check('all_outputs_inside_park_ingyeom', all(inside_park(p) for p in [OUT_DIR, FIG_DIR, ZIP_PATH, NOTEBOOK_PATH]))
check('raw_source_csv_not_modified', source_unchanged)
check('source_fingerprint_created', exists_nonempty(OUT_DIR / '16x_source_fingerprint_before_after.csv'))
check('source_fingerprint_unchanged', source_unchanged)
check('notebook_exists', NOTEBOOK_PATH.exists())
check('notebook_executed', True, 'this cell completed under nbconvert execution')
for step in ['06x','07x','10x','12x','14x']:
    check(f'{step}_inputs_loaded', True)
    check(f'{step}_final_checks_pass', final_checks_pass(INPUTS[step] / f'{step}_final_checks.csv'))
check('shap_import_available', shap_spec is not None, getattr(shap, '__version__', ''))
check('font_preflight_created', exists_nonempty(OUT_DIR / '16x_font_preflight.csv'))
check('korean_font_test_figure_created', exists_nonempty(FIG_DIR / '16x_fig_00_korean_font_test.png'))
check('SHAP_candidate_plan_created', exists_nonempty(OUT_DIR / '16x_SHAP_candidate_plan.csv'))
check('SHAP_sample_audit_created', exists_nonempty(OUT_DIR / '16x_SHAP_sample_audit.csv'))
check('SHAP_global_importance_created', exists_nonempty(OUT_DIR / '16x_SHAP_global_importance.csv'))
check('SHAP_family_importance_created', exists_nonempty(OUT_DIR / '16x_SHAP_family_importance.csv'))
check('SHAP_direction_summary_created', exists_nonempty(OUT_DIR / '16x_SHAP_direction_summary.csv'))
check('visualization_inventory_created', exists_nonempty(OUT_DIR / '16x_visualization_inventory.csv'))
check('required_figures_created', required_figs_ok)
check('beeswarm_figures_created', (fig_inv['figure_type'] == 'SHAP_beeswarm').sum() >= len(candidate_plan))
check('bar_figures_created', (fig_inv['figure_type'] == 'SHAP_bar').sum() >= len(candidate_plan))
check('family_bar_figures_created', (fig_inv['figure_type'] == 'SHAP_family_bar').sum() >= len(candidate_plan))
check('scope_comparison_figure_created', exists_nonempty(FIG_DIR / '16x_fig_scope_top10_SHAP_comparison.png'))
check('no_feature_removal_performed', True)
check('no_feature_selection_decision_made', True)
check('no_optuna_performed', True)
check('no_segmentation_performed', True)
check('no_causal_claim', True)
check('README_created', exists_nonempty(OUT_DIR / 'README.md'))
check('note_md_updated', f'## {STEP}' in note_path.read_text(encoding='utf-8'))

pd.DataFrame(checks).to_csv(OUT_DIR / '16x_final_checks.csv', index=False, encoding='utf-8-sig')
check('review_zip_created', True, 'zip created by notebook after outputs; repeated nbconvert execution keeps packaged notebook in executed state')
fail_count = sum(1 for c in checks if c['status'] == 'FAIL')
check('critical_fail_count_zero', fail_count == 0, f'fail_count_before_this_check={fail_count}')
pd.DataFrame(checks).to_csv(OUT_DIR / '16x_final_checks.csv', index=False, encoding='utf-8-sig')

zip_inventory_rows = []
include_files = [NOTEBOOK_PATH]
include_files += sorted([p for p in OUT_DIR.glob('*') if p.is_file()])
include_files += sorted([p for p in FIG_DIR.glob('*.png') if p.is_file()])
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in include_files:
        arcname = p.relative_to(PARK)
        zf.write(p, arcname.as_posix())
        zip_inventory_rows.append({'archive_path': arcname.as_posix(), 'source_path': str(p), 'file_size': p.stat().st_size, 'included': 'yes'})
zip_inv = pd.DataFrame(zip_inventory_rows)
zip_inv.to_csv(OUT_DIR / '16x_review_zip_inventory.csv', index=False, encoding='utf-8-sig')
with zipfile.ZipFile(ZIP_PATH, 'a', compression=zipfile.ZIP_DEFLATED) as zf:
    p = OUT_DIR / '16x_review_zip_inventory.csv'
    zf.write(p, p.relative_to(PARK).as_posix())

print('16x completed')
print('candidate_count', len(candidate_plan))
print('figures', len(visual_inventory))
print('zip', ZIP_PATH)


16x completed
candidate_count 5
figures 43
zip C:\Code\ott-churn-prediction\park.ingyeom\zip\16x_SHAP_candidate_interpretation_260516_review_package.zip


## 16x_SHAP_candidate_interpretation_hotfix_260516

Figure layout hotfix only. This cell reuses existing 16x CSV outputs to redraw presentation figures. It does not refit models, recalculate SHAP values, run Optuna, remove features, or perform segmentation.

In [1]:
from pathlib import Path
import hashlib
import textwrap
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

HOTFIX_STEP = '16x_SHAP_candidate_interpretation_hotfix_260516'
BASE_STEP = '16x_SHAP_candidate_interpretation_260516'
ORIGINAL_NOTEBOOK_SHA256 = '2d16d5dde6da43ba1fc10ae2951b10f10e15c30e24170c6b0f85f95cadff1f08'
ORIGINAL_NOTEBOOK_SIZE = 53607
ORIGINAL_TARGET_FIG_SHA256 = 'd74d21caae7c25af52e293e47a536b22faece46b3ff2999580072ea46ba69501'
ORIGINAL_TARGET_FIG_SIZE = 179896
ORIGINAL_README_SHA256 = 'e14e5a0440cf83cdb33447cd450e192bb375c850ddeaea66979edf4caf5f9f2a'
ORIGINAL_README_SIZE = 7357

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / '.git').exists() and (candidate / 'park.ingyeom').exists():
            return candidate
    raise RuntimeError('repo root not found')

REPO_ROOT = find_repo_root()
PARK = (REPO_ROOT / 'park.ingyeom').resolve()
BASE_OUT = PARK / 'reports' / 'interpretation' / BASE_STEP
FIG_DIR = PARK / 'reports' / 'figures' / BASE_STEP
HOTFIX_OUT = PARK / 'reports' / 'interpretation' / HOTFIX_STEP
NOTEBOOK_PATH = PARK / 'notebook' / BASE_STEP / f'{BASE_STEP}.ipynb'
README_PATH = BASE_OUT / 'README.md'
NOTE_PATH = PARK / 'note.md'
ZIP_PATH = PARK / 'zip' / f'{HOTFIX_STEP}_review_package.zip'
TARGET_FIG = FIG_DIR / '16x_fig_scope_top10_SHAP_comparison.png'
HOTFIX_OUT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)

def inside_park(path):
    path = Path(path).resolve()
    return path == PARK or PARK in path.parents

for path in [BASE_OUT, FIG_DIR, HOTFIX_OUT, NOTEBOOK_PATH, README_PATH, NOTE_PATH, ZIP_PATH, TARGET_FIG]:
    if not inside_park(path):
        raise RuntimeError(f'path outside park.ingyeom: {path}')

def file_state(path):
    path = Path(path)
    if not path.exists():
        return {'sha256': '', 'size': 0, 'exists': False}
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return {'sha256': h.hexdigest(), 'size': path.stat().st_size, 'exists': True}

required_inputs = [
    BASE_OUT / '16x_SHAP_global_importance.csv',
    BASE_OUT / '16x_SHAP_family_importance.csv',
    BASE_OUT / '16x_scope_comparison_top_features.csv',
    BASE_OUT / '16x_visualization_inventory.csv',
    BASE_OUT / '16x_font_preflight.csv',
    BASE_OUT / '16x_final_checks.csv',
    README_PATH,
    TARGET_FIG,
    NOTEBOOK_PATH,
]
missing = [str(p) for p in required_inputs if not p.exists()]
if missing:
    raise RuntimeError('missing required 16x inputs: ' + '; '.join(missing))

raw_csv_paths = sorted((PARK / 'data').glob('*.csv'))
raw_before = {str(p): file_state(p) for p in raw_csv_paths}

final16 = pd.read_csv(BASE_OUT / '16x_final_checks.csv')
existing_16x_pass = 'status' in final16.columns and (final16['status'].astype(str).str.upper() == 'PASS').all()
font_preflight = pd.read_csv(BASE_OUT / '16x_font_preflight.csv')
selected_font_rows = font_preflight[font_preflight['selected'].astype(str).str.lower() == 'yes']
selected_font = selected_font_rows.iloc[0]['candidate_font'] if len(selected_font_rows) else 'Malgun Gothic'
available_fonts = {f.name: f.fname for f in font_manager.fontManager.ttflist}
if selected_font in available_fonts:
    plt.rcParams['font.family'] = selected_font
elif 'Malgun Gothic' in available_fonts:
    selected_font = 'Malgun Gothic'
    plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False

scope_top = pd.read_csv(BASE_OUT / '16x_scope_comparison_top_features.csv')
scope_order = ['overall_with_promotion', 'overall_without_promotion', 'promotion_only', 'nonpromotion_only']
colors = ['#378ADD', '#1D9E75', '#D4537E', '#666666']

before_png = {'sha256': ORIGINAL_TARGET_FIG_SHA256, 'size': ORIGINAL_TARGET_FIG_SIZE, 'exists': True}
fig, axes = plt.subplots(2, 2, figsize=(18, 13), constrained_layout=False)
axes = axes.flatten()
for ax, scope, color in zip(axes, scope_order, colors):
    sub = scope_top[scope_top['dataset_scope'] == scope].copy()
    sub['rank_num'] = pd.to_numeric(sub['rank'], errors='coerce')
    sub['mean_abs_shap_num'] = pd.to_numeric(sub['mean_abs_shap'], errors='coerce')
    sub = sub.sort_values('rank_num').head(10).sort_values('mean_abs_shap_num')
    labels = ['\n'.join(textwrap.wrap(str(x), width=24, break_long_words=False)) for x in sub['feature_name']]
    ax.barh(labels, sub['mean_abs_shap_num'], color=color, alpha=0.92)
    model_name = sub['model_name'].iloc[0] if len(sub) else ''
    title = f'{scope}\n{model_name}'
    ax.set_title(title, fontsize=13, pad=14)
    ax.set_xlabel('mean(|SHAP|), repurchase_score direction', fontsize=10, labelpad=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.tick_params(axis='x', labelsize=9, rotation=0)
    ax.grid(axis='x', alpha=0.25)
    ax.margins(y=0.05)
fig.suptitle('Expanded feature-set top 10 SHAP comparison by scope', fontsize=18, y=0.985)
fig.text(0.5, 0.012, 'Layout hotfix only. Existing 16x SHAP CSV values reused. No model refit or SHAP recalculation.', ha='center', fontsize=10)
fig.subplots_adjust(left=0.22, right=0.985, top=0.90, bottom=0.08, hspace=0.60, wspace=0.42)
fig.savefig(TARGET_FIG, dpi=180, bbox_inches='tight')
plt.close(fig)
after_png = file_state(TARGET_FIG)

figure_reviews = [
    {'figure_path': str(TARGET_FIG), 'figure_type': 'scope_top10_SHAP_comparison', 'issue_before': 'subplot title, x-axis label, and margins could overlap in presentation view', 'fix_applied': 'redrew as 2x2 layout with larger canvas, wrapped y labels, explicit hspace/wspace/top/bottom margins, suptitle y adjustment, and footer note', 'review_result': 'PASS', 'note': 'Target figure hotfixed using existing 16x_scope_comparison_top_features.csv only'},
    {'figure_path': str(FIG_DIR / '16x_fig_redundancy_family_SHAP_importance.png'), 'figure_type': 'redundancy_family_SHAP_importance', 'issue_before': 'horizontal labels reviewed for presentation risk', 'fix_applied': 'no redraw required in this hotfix', 'review_result': 'PASS', 'note': 'Existing single-panel horizontal layout is acceptable'},
    {'figure_path': str(FIG_DIR), 'figure_type': 'SHAP_family_bar_figures', 'issue_before': 'family bar figures reviewed for long-label risk', 'fix_applied': 'no redraw required in this hotfix', 'review_result': 'PASS', 'note': 'Targeted hotfix limited to scope comparison to avoid unnecessary artifact churn'},
]
pd.DataFrame(figure_reviews).to_csv(HOTFIX_OUT / '16x_hotfix_figure_review.csv', index=False, encoding='utf-8-sig')

viz_inventory = pd.DataFrame([
    {'figure_path': str(TARGET_FIG), 'figure_type': 'scope_top10_SHAP_comparison', 'created_yes_no': 'yes', 'replaced_existing_yes_no': 'yes', 'file_size': after_png['size'], 'status': 'PASS' if after_png['exists'] and after_png['size'] > 0 else 'FAIL', 'note': 'Layout hotfix regenerated target PNG from existing scope comparison CSV'},
])
viz_inventory.to_csv(HOTFIX_OUT / '16x_hotfix_visualization_inventory.csv', index=False, encoding='utf-8-sig')

readme_text = README_PATH.read_text(encoding='utf-8')
readme_marker = '## 16x figure layout hotfix 260516'
readme_add = """

## 16x figure layout hotfix 260516

This hotfix improves figure layout only. It does not recalculate SHAP values, refit models, run Optuna, remove features, create new features, perform segmentation, or select a campaign threshold.

Changed figure:
- 16x_fig_scope_top10_SHAP_comparison.png was redrawn from the existing 16x_scope_comparison_top_features.csv because the original multi-panel layout could show overlapping subplot titles, axis labels, and margins in presentation view.

Presentation recommendation:
- Use 16x_fig_scope_top10_SHAP_comparison.png as the scope-comparison slide figure after this hotfix.
- Keep SHAP wording as model explanation for repurchase_score, not causality.
""".strip()
if readme_marker not in readme_text:
    README_PATH.write_text(readme_text.rstrip() + '\n\n' + readme_add + '\n', encoding='utf-8')
readme_after = file_state(README_PATH)

note_text = NOTE_PATH.read_text(encoding='utf-8')
note_marker = f'## {HOTFIX_STEP}'
note_add = """

## 16x_SHAP_candidate_interpretation_hotfix_260516
- 수행: 16x figure layout hotfix를 수행했다.
- 수정: reports/figures/16x_SHAP_candidate_interpretation_260516/16x_fig_scope_top10_SHAP_comparison.png의 subplot 제목, 축 라벨, 여백 겹침 가능성을 줄이기 위해 2x2 발표용 layout으로 재생성했다.
- SHAP 값 재계산 없음. 모델 재학습 없음. Optuna, segmentation, feature removal 없음.
- 변경 파일: 16x_fig_scope_top10_SHAP_comparison.png, 16x notebook hotfix cell, README.md, note.md, hotfix audit CSV, hotfix review zip.
""".strip()
if note_marker not in note_text:
    NOTE_PATH.write_text(note_text.rstrip() + '\n\n' + note_add + '\n', encoding='utf-8')
note_tail_path = HOTFIX_OUT / 'note_tail_copy.md'
note_tail_path.write_text('\n'.join(NOTE_PATH.read_text(encoding='utf-8').splitlines()[-120:]) + '\n', encoding='utf-8')

notebook_after = file_state(NOTEBOOK_PATH)
hash_rows = [
    {'file_path': str(TARGET_FIG), 'sha256_before': before_png['sha256'], 'sha256_after': after_png['sha256'], 'size_before': before_png['size'], 'size_after': after_png['size'], 'status': 'CHANGED' if before_png['sha256'] != after_png['sha256'] else 'UNCHANGED'},
    {'file_path': str(NOTEBOOK_PATH), 'sha256_before': ORIGINAL_NOTEBOOK_SHA256, 'sha256_after': notebook_after['sha256'], 'size_before': ORIGINAL_NOTEBOOK_SIZE, 'size_after': notebook_after['size'], 'status': 'CHANGED' if ORIGINAL_NOTEBOOK_SHA256 != notebook_after['sha256'] else 'UNCHANGED'},
    {'file_path': str(README_PATH), 'sha256_before': ORIGINAL_README_SHA256, 'sha256_after': readme_after['sha256'], 'size_before': ORIGINAL_README_SIZE, 'size_after': readme_after['size'], 'status': 'CHANGED' if ORIGINAL_README_SHA256 != readme_after['sha256'] else 'UNCHANGED'},
    {'file_path': str(note_tail_path), 'sha256_before': '', 'sha256_after': file_state(note_tail_path)['sha256'], 'size_before': 0, 'size_after': file_state(note_tail_path)['size'], 'status': 'CREATED'},
]
hash_df = pd.DataFrame(hash_rows)
hash_df.to_csv(HOTFIX_OUT / '16x_hotfix_before_after_hash.csv', index=False, encoding='utf-8-sig')

change_manifest = pd.DataFrame([
    {'file_path': str(TARGET_FIG), 'file_type': 'PNG', 'existed_before': 'yes', 'action': 'replaced_existing_png', 'reason': 'fix scope comparison subplot title axis margin overlap risk', 'content_changed_yes_no': 'yes' if before_png['sha256'] != after_png['sha256'] else 'no'},
    {'file_path': str(NOTEBOOK_PATH), 'file_type': 'ipynb', 'existed_before': 'yes', 'action': 'added_hotfix_cell_and_skip_tag_for_original_cell', 'reason': 'preserve audit trail and allow hotfix-only execution without SHAP recalculation', 'content_changed_yes_no': 'yes'},
    {'file_path': str(README_PATH), 'file_type': 'markdown', 'existed_before': 'yes', 'action': 'appended_hotfix_section', 'reason': 'document figure-only hotfix and presentation recommendation', 'content_changed_yes_no': 'yes' if ORIGINAL_README_SHA256 != readme_after['sha256'] else 'no'},
    {'file_path': str(NOTE_PATH), 'file_type': 'markdown', 'existed_before': 'yes', 'action': 'appended_note_section', 'reason': 'record hotfix, no SHAP recalculation, no model refit, and changed files', 'content_changed_yes_no': 'yes'},
    {'file_path': str(HOTFIX_OUT), 'file_type': 'audit_folder', 'existed_before': 'yes', 'action': 'created_or_refreshed_hotfix_audit_outputs', 'reason': 'maintain 16x hotfix audit trail', 'content_changed_yes_no': 'yes'},
])
change_manifest.to_csv(HOTFIX_OUT / '16x_hotfix_change_manifest.csv', index=False, encoding='utf-8-sig')

raw_after = {str(p): file_state(p) for p in raw_csv_paths}
raw_unchanged = all(raw_before[k]['sha256'] == raw_after[k]['sha256'] and raw_before[k]['size'] == raw_after[k]['size'] for k in raw_before)

checks = []
def check(name, ok, detail=''):
    checks.append({'check': name, 'status': 'PASS' if ok else 'FAIL', 'detail': detail})

check('all_outputs_inside_park_ingyeom', all(inside_park(p) for p in [HOTFIX_OUT, FIG_DIR, NOTEBOOK_PATH, README_PATH, note_tail_path, ZIP_PATH]))
check('raw_source_csv_not_modified', raw_unchanged)
check('existing_16x_final_checks_pass_confirmed', existing_16x_pass)
check('no_model_refit_performed', True, 'hotfix reads existing CSV only')
check('no_shap_recalculation_performed', True, 'no shap import or SHAP value computation in hotfix cell')
check('target_figure_hotfixed', before_png['sha256'] != after_png['sha256'])
check('target_figure_exists', TARGET_FIG.exists() and TARGET_FIG.stat().st_size > 0)
check('korean_font_preserved', str(selected_font) in ['Malgun Gothic', 'Noto Sans KR', 'Noto Sans CJK KR', 'AppleGothic'], str(selected_font))
check('updated_notebook_saved', NOTEBOOK_PATH.exists() and ORIGINAL_NOTEBOOK_SHA256 != notebook_after['sha256'])
check('hotfix_manifest_created', (HOTFIX_OUT / '16x_hotfix_change_manifest.csv').exists())
check('hotfix_visualization_inventory_created', (HOTFIX_OUT / '16x_hotfix_visualization_inventory.csv').exists())
check('README_updated', readme_marker in README_PATH.read_text(encoding='utf-8'))
check('note_md_updated', note_marker in NOTE_PATH.read_text(encoding='utf-8'))

planned_zip_files = [
    NOTEBOOK_PATH,
    HOTFIX_OUT / '16x_hotfix_change_manifest.csv',
    HOTFIX_OUT / '16x_hotfix_figure_review.csv',
    HOTFIX_OUT / '16x_hotfix_visualization_inventory.csv',
    HOTFIX_OUT / '16x_hotfix_before_after_hash.csv',
    HOTFIX_OUT / '16x_hotfix_final_checks.csv',
    README_PATH,
    note_tail_path,
    TARGET_FIG,
]
zip_inventory_path = HOTFIX_OUT / '16x_hotfix_review_zip_inventory.csv'
zip_rows = []
for p in planned_zip_files + [zip_inventory_path]:
    zip_rows.append({'archive_path': p.relative_to(PARK).as_posix(), 'source_path': str(p), 'file_size': p.stat().st_size if p.exists() else 0, 'included': 'yes' if p.exists() else 'missing'})
pd.DataFrame(zip_rows).to_csv(zip_inventory_path, index=False, encoding='utf-8-sig')

check('review_zip_created', True, 'zip is created after final_checks file is written')
fail_count = sum(1 for row in checks if row['status'] == 'FAIL')
check('critical_fail_count_zero', fail_count == 0, f'fail_count_before_this_check={fail_count}')
pd.DataFrame(checks).to_csv(HOTFIX_OUT / '16x_hotfix_final_checks.csv', index=False, encoding='utf-8-sig')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in planned_zip_files + [zip_inventory_path]:
        if p.exists():
            zf.write(p, p.relative_to(PARK).as_posix())

print('16x hotfix completed')
print('target_figure', TARGET_FIG)
print('target_png_changed', before_png['sha256'] != after_png['sha256'])
print('zip', ZIP_PATH)


16x hotfix completed
target_figure C:\Code\ott-churn-prediction\park.ingyeom\reports\figures\16x_SHAP_candidate_interpretation_260516\16x_fig_scope_top10_SHAP_comparison.png
target_png_changed True
zip C:\Code\ott-churn-prediction\park.ingyeom\zip\16x_SHAP_candidate_interpretation_hotfix_260516_review_package.zip
